# HrudayAI — Phase 1: Hindi & English Audio-to-Text

## Scope
- **Target Languages:** Hindi (`hi`) and English (`en`) only.
- **Models:** Free, open-source speech recognition models configurable via `phase1_voice/configs/asr_models.json` (no paid APIs).
- **Design:** Audio loading/preprocessing (`phase1_voice.src.audio_utils`) is strictly separated from speech transcription (`phase1_voice.src.transcription`).
- **Execution:** Heavy ASR model loading and inference run inside Google Colab (GPU preferred, CPU fallback supported).

> **Privacy Reminder:** Test audio uploaded in Colab stays in the ephemeral `/content/` runtime only. Never commit audio clips or patient data to GitHub.

## 1. Environment & Dependency Setup

1. Detect Google Colab vs. local runtime.
2. Locate or clone the `HrudayAI` repository (no hardcoded Windows paths).
3. Install Phase 1 open-source dependencies in Colab when needed and verify `ffmpeg` + device (`cuda` or `cpu`).

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_ROOT = Path("/content/HrudayAI")
    if not REPO_ROOT.exists():
        # Optional: if you saved GITHUB_TOKEN in Colab Secrets, clone automatically
        token = None
        try:
            from google.colab import userdata  # type: ignore
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = os.environ.get("GITHUB_TOKEN")

        if token:
            subprocess.check_call([
                "git", "clone",
                f"https://{token}@github.com/itsakshatsinghh/HrudayAI.git",
                str(REPO_ROOT),
            ])
        else:
            raise RuntimeError(
                "Repository not found at /content/HrudayAI. "
                "Clone it in a cell first (e.g., `!git clone https://<TOKEN>@github.com/itsakshatsinghh/HrudayAI.git /content/HrudayAI`) "
                "or set a Colab Secret named GITHUB_TOKEN."
            )
    os.chdir(REPO_ROOT)
else:
    # Resolve repo root dynamically relative to this notebook
    cwd = Path.cwd().resolve()
    REPO_ROOT = cwd.parent.parent if cwd.name == "notebooks" else cwd

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REQ_FILE = REPO_ROOT / "phase1_voice" / "requirements.txt"
if not REQ_FILE.exists():
    raise FileNotFoundError(
        f"Missing requirements file at {REQ_FILE}. "
        "If /content/HrudayAI was cloned earlier, run `!git -C /content/HrudayAI pull origin main`."
    )

print(f"Running in Colab : {IN_COLAB}")
print(f"Repo root        : {REPO_ROOT}")
print(f"Working dir      : {Path.cwd()}")
print(f"Requirements     : {REQ_FILE} (exists={REQ_FILE.exists()})")
print(f"ffmpeg available : {shutil.which('ffmpeg') is not None}")

In [ ]:
# Install Phase 1 dependencies in Google Colab from phase1_voice/requirements.txt
INSTALL_DEPS = IN_COLAB

if INSTALL_DEPS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)])
    importlib.invalidate_caches()
    print(f"Installed Phase 1 dependencies from {REQ_FILE}.")
else:
    print(f"Skipping pip install (set INSTALL_DEPS = True to install {REQ_FILE}).")

from phase1_voice.src import resolve_device

device, compute_type = resolve_device("auto")
print(f"Resolved device  : {device} (compute_type={compute_type})")

## 2. Configurable Open-Source Model Selection

Models are defined in `phase1_voice/configs/asr_models.json` so we can switch and benchmark open-source models cleanly without changing notebook logic.

In [ ]:
from phase1_voice.src import get_model_config, list_available_models, load_asr_model

available_models = list_available_models()
print("Available Phase 1 open-source ASR models:")
for key, cfg in available_models.items():
    print(f"  - {key}: {cfg.display_name} [{cfg.backend} | {cfg.model_id}] langs={cfg.supported_languages}")

# Change SELECTED_MODEL_KEY to compare models (e.g. 'faster_whisper_base', 'faster_whisper_small', 'hf_whisper_small')
SELECTED_MODEL_KEY = "faster_whisper_small"
selected_cfg = get_model_config(SELECTED_MODEL_KEY)
print(f"\nSelected model: {selected_cfg.key} -> {selected_cfg.model_id}")

# Load model once per runtime (cached inside load_asr_model to avoid repeated downloads)
LOAD_MODEL = IN_COLAB
MODEL_BUNDLE = globals().get("MODEL_BUNDLE")
if LOAD_MODEL:
    try:
        MODEL_BUNDLE = load_asr_model(model_key=SELECTED_MODEL_KEY, device="auto")
        print(f"Model ready on device='{MODEL_BUNDLE['device']}' ({MODEL_BUNDLE['compute_type']}).")
    except Exception as exc:
        MODEL_BUNDLE = None
        print(f"Model loading failed: {exc}")
else:
    print("Model loading skipped outside Colab (set LOAD_MODEL = True to load weights).")

## 3. Audio Upload & Preprocessing

Upload temporary Hindi and/or English audio clips (`.wav`, `.mp3`, `.flac`, `.ogg`, `.m4a`). Audio loading and preprocessing are handled separately from ASR transcription via `phase1_voice.src.audio_utils`.

In [ ]:
from pathlib import Path

HINDI_AUDIO_PATH = None
ENGLISH_AUDIO_PATH = None

if IN_COLAB:
    from google.colab import files  # type: ignore

    print("Upload a HINDI test audio file (or cancel to skip):")
    hi_uploaded = files.upload()
    if hi_uploaded:
        HINDI_AUDIO_PATH = Path(next(iter(hi_uploaded.keys()))).resolve()

    print("Upload an ENGLISH test audio file (or cancel to skip):")
    en_uploaded = files.upload()
    if en_uploaded:
        ENGLISH_AUDIO_PATH = Path(next(iter(en_uploaded.keys()))).resolve()
else:
    print("Local environment: set HINDI_AUDIO_PATH / ENGLISH_AUDIO_PATH manually if testing locally.")

print(f"Hindi audio path  : {HINDI_AUDIO_PATH}")
print(f"English audio path: {ENGLISH_AUDIO_PATH}")

In [ ]:
from IPython.display import Audio, display
from phase1_voice.src import (
    AudioLoadError,
    EmptyAudioError,
    UnsupportedAudioFormatError,
    get_audio_metadata,
    load_audio,
)


def prepare_audio_clip(audio_path: Path | None, label: str):
    if audio_path is None:
        print(f"[{label}] No file provided; skipping.")
        return None, None
    try:
        waveform, sr = load_audio(audio_path, target_sr=16000, mono=True, normalize=True)
        metadata = get_audio_metadata(waveform, sr)
        print(f"[{label}] Preprocessed audio metadata: {metadata}")
        display(Audio(waveform, rate=sr))
        return waveform, sr
    except UnsupportedAudioFormatError as exc:
        print(f"[{label}] Unsupported format error: {exc}")
    except EmptyAudioError as exc:
        print(f"[{label}] Empty/silent audio error: {exc}")
    except (FileNotFoundError, AudioLoadError) as exc:
        print(f"[{label}] Audio loading failed: {exc}")
    return None, None


hi_waveform, hi_sr = prepare_audio_clip(HINDI_AUDIO_PATH, "Hindi")
en_waveform, en_sr = prepare_audio_clip(ENGLISH_AUDIO_PATH, "English")

## 4. Hindi & English Transcription Tests

Run speech-to-text inference separately for Hindi (`hi`), English (`en`), and automatic Hindi/English language detection (`language=None`), displaying the detected language and transcription output.

In [ ]:
import json
import time
from phase1_voice.src import TranscriptionError, UnsupportedLanguageError, transcribe_audio


def run_transcription_test(waveform, sample_rate, language_hint, label: str):
    if MODEL_BUNDLE is None:
        print(f"[{label}] MODEL_BUNDLE is not loaded. Run Section 2 in Colab first.")
        return None
    if waveform is None:
        print(f"[{label}] No preprocessed audio available; upload a clip in Section 3.")
        return None

    start = time.perf_counter()
    try:
        result = transcribe_audio(
            waveform=waveform,
            sample_rate=sample_rate,
            model_bundle=MODEL_BUNDLE,
            language=language_hint,
        )
        elapsed = round(time.perf_counter() - start, 3)
        print(f"=== {label} Result (elapsed={elapsed}s) ===")
        print(f"Detected Language : {result['detected_language']} (prob={result['language_probability']})")
        print(f"Model / Device    : {result['model_key']} ({result['model_id']}) on {result['device']}")
        print(f"Transcription     : {result['text']}")
        if result["warnings"]:
            print(f"Warnings          : {result['warnings']}")
        return result
    except UnsupportedLanguageError as exc:
        print(f"[{label}] Language error: {exc}")
    except TranscriptionError as exc:
        print(f"[{label}] Transcription error: {exc}")
    return None

In [ ]:
# 4A. Hindi Transcription Test (explicit language='hi' + auto-detection check)
hindi_result = run_transcription_test(hi_waveform, hi_sr, language_hint="hi", label="Hindi (explicit)")
hindi_auto_result = run_transcription_test(hi_waveform, hi_sr, language_hint=None, label="Hindi (auto-detect)")

In [ ]:
# 4B. English Transcription Test (explicit language='en' + auto-detection check)
english_result = run_transcription_test(en_waveform, en_sr, language_hint="en", label="English (explicit)")
english_auto_result = run_transcription_test(en_waveform, en_sr, language_hint=None, label="English (auto-detect)")

## 5. Structured Output Display & Error Handling Verification

In [ ]:
summary_pairs = [
    ("Hindi (explicit)", globals().get("hindi_result")),
    ("Hindi (auto-detect)", globals().get("hindi_auto_result")),
    ("English (explicit)", globals().get("english_result")),
    ("English (auto-detect)", globals().get("english_auto_result")),
]
for name, payload in summary_pairs:
    if payload is not None:
        print(f"\n--- {name} ---")
        print(f"Detected Language : {payload['detected_language']} (prob={payload['language_probability']})")
        print(f"Recognized Text   : {payload['text']}")
        print(json.dumps(payload, indent=2, ensure_ascii=False))

# Quick error-handling smoke check (empty waveform & unsupported language)
import numpy as np

try:
    transcribe_audio(np.array([], dtype=np.float32), sample_rate=16000, model_bundle=MODEL_BUNDLE)
except TranscriptionError as exc:
    print(f"Empty waveform check passed: {exc}")

## 6. Model Performance & Error Log (Fill After Colab Runs)

Record observed Colab results here (do **not** paste sensitive patient audio transcripts):

| Date | Colab Runtime (CPU/T4) | Model Key | Audio Duration (s) | Language (`hi`/`en`) | Detected Lang & Prob | Latency (s) | Accuracy Notes / Errors Observed |
| --- | --- | --- | --- | --- | --- | --- | --- |
| _YYYY-MM-DD_ | _T4 / CPU_ | `faster_whisper_small` | _0.0_ | `hi` | _hi (0.00)_ | _0.0_ | _Pending Colab test_ |
| _YYYY-MM-DD_ | _T4 / CPU_ | `faster_whisper_small` | _0.0_ | `en` | _en (0.00)_ | _0.0_ | _Pending Colab test_ |
